# Diabetes Prediction with Machine Learning + Explainable AI

Based on: **"Towards Transparent and Accurate Diabetes Prediction Using Machine Learning and Explainable Artificial Intelligence"**

| Step | What we do |
|------|------------|
| 1 | Install & import libraries |
| 2 | Load dataset |
| 3 | Pre-process (impute → scale → SMOTE) |
| 4 | Train / Validation / Test split |
| 5 | Train 7 individual ML models |
| 6 | Build Ensemble model (RF + XGBoost + LightGBM) |
| 7 | Compare all models — Table II |
| 8 | SHAP explanations |
| 9 | Permutation Importance |
| 10 | EBM (Explainable Boosting Machine) |
| 11 | LIME explanations |
| 12 | Partial Dependence Plots |
| 13 | Anchor explanations |
| 14 | Counterfactual explanations |
| 15 | Explainability Metrics — Table IV |

> **Dataset needed:** Download `diabetes_binary_health_indicators_BRFSS2015.csv` from  
> https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset  
> and place it in the same folder as this notebook.

## Step 1 — Install Libraries
Run this cell once. You can skip it if the libraries are already installed.

In [ ]:
# Run once to install all required libraries
%pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost lightgbm shap lime interpret alibi dice-ml --quiet

## Step 2 — Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# --- basic tools ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# --- scikit-learn ---
from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     StratifiedKFold)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, classification_report)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# --- handle class imbalance ---
from imblearn.over_sampling import SMOTE

# --- boosting models ---
import xgboost as xgb
import lightgbm as lgb

# --- explainability ---
import shap
import lime
import lime.lime_tabular
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show as ebm_show

print("All libraries imported successfully!")

## Step 3 — Load Dataset

The dataset has **253,680 patients** and **22 health features** collected by the CDC BRFSS survey.  
The target column `Diabetes_binary` is:  
- `0` = Non-Diabetic  
- `1` = Diabetic

In [ ]:
CSV_FILE = "/kaggle/input/notebooks/alexteboul/diabetes-health-indicators-dataset-notebook/diabetes_binary_health_indicators_BRFSS2015.csv"
TARGET   = "Diabetes_binary"

df = pd.read_csv(CSV_FILE)

print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df[TARGET].value_counts())
print(f"\nClass percentages:")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))
print(f"\nMissing values: {df.isnull().sum().sum()}")
df.head()

In [ ]:
# Quick look at the class imbalance
fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax,
    color=["steelblue", "tomato"], edgecolor="black")
ax.set_xticklabels(["Non-Diabetic (0)", "Diabetic (1)"], rotation=0)
ax.set_title("Class Distribution (before SMOTE)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Step 4 — Pre-processing

Three things to do:
1. **Impute** missing values with the **median** (robust to outliers in medical data)
2. **Scale** features to mean = 0, std = 1 so all features contribute equally
3. **SMOTE** — create synthetic minority-class samples to fix the class imbalance  
   (applied only on the training set, never on val/test)

In [ ]:
# --- 4a. Separate features and target ---
X = df.drop(TARGET, axis=1)
y = df[TARGET]
feature_names = X.columns.tolist()
print(f"Features ({len(feature_names)}): {feature_names}")

In [ ]:
# --- 4b. Fill missing values with MEDIAN ---
# Median is used because it is not affected by extreme outliers
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=feature_names)
print(f"Missing values after imputation: {X.isnull().sum().sum()}")

In [ ]:
# --- 4c. Scale features (StandardScaler → mean=0, std=1) ---
# This makes sure BMI, blood pressure, cholesterol etc. are on
# the same scale so no single feature dominates because of its unit.
scaler  = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)
print("After scaling — mean and std of first 3 features:")
print(X_scaled[feature_names[:3]].agg(["mean", "std"]).round(4))

## Step 5 — Train / Validation / Test Split

| Set | Size | Purpose |
|-----|------|---------|
| Train | 70% | Model learns from this |
| Validation | 15% | Tune hyperparameters |
| Test | 15% | Final unbiased evaluation |

**Stratified** sampling keeps the same 87% / 13% class ratio in every split.

In [ ]:
# First take out the 15% test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42, stratify=y
)

# From the remaining 85%, carve out validation (≈15% of total)
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=val_ratio, random_state=42, stratify=y_train_val
)

total = len(X_scaled)
print(f"Train:      {len(X_train):>6}  ({len(X_train)/total*100:.1f}%)")
print(f"Validation: {len(X_val):>6}  ({len(X_val)/total*100:.1f}%)")
print(f"Test:       {len(X_test):>6}  ({len(X_test)/total*100:.1f}%)")

In [ ]:
# --- Apply SMOTE on TRAINING set only ---
# SMOTE creates new synthetic diabetic samples so both classes are equal.
# Rule: NEVER apply SMOTE to validation or test — that would fake the results.
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
X_train_bal = pd.DataFrame(X_train_bal, columns=feature_names)

print(f"Before SMOTE → {dict(y_train.value_counts())}")
print(f"After  SMOTE → {dict(pd.Series(y_train_bal).value_counts())}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
y_train.value_counts().plot(kind="bar", ax=axes[0], color=["steelblue","tomato"],
                             edgecolor="black", title="Before SMOTE")
pd.Series(y_train_bal).value_counts().plot(kind="bar", ax=axes[1],
    color=["steelblue","tomato"], edgecolor="black", title="After SMOTE")
for ax in axes:
    ax.set_xticklabels(["Non-Diabetic", "Diabetic"], rotation=0)
plt.tight_layout()
plt.show()

## Step 6 — Train 7 Individual ML Models

We use **RandomizedSearchCV with 3-fold cross-validation** to find the best settings for each model automatically.

In [ ]:
# Helper: evaluate a trained model on val and test sets
def evaluate_model(model, name, X_v, y_v, X_te, y_te):
    val_pred  = model.predict(X_v)
    test_pred = model.predict(X_te)

    if hasattr(model, "predict_proba"):
        val_prob  = model.predict_proba(X_v)[:, 1]
        test_prob = model.predict_proba(X_te)[:, 1]
    else:
        val_prob  = model.decision_function(X_v)
        test_prob = model.decision_function(X_te)

    return {
        "Model":             name,
        "Val Accuracy (%)":  round(accuracy_score(y_v,  val_pred)  * 100, 2),
        "Test Accuracy (%)": round(accuracy_score(y_te, test_pred) * 100, 2),
        "Val ROC-AUC":       round(roc_auc_score(y_v,  val_prob),  3),
        "Test ROC-AUC":      round(roc_auc_score(y_te, test_prob), 3),
        "Precision":         round(precision_score(y_te, test_pred, zero_division=0), 2),
        "Recall":            round(recall_score(y_te, test_pred),   2),
        "F1-Score":          round(f1_score(y_te, test_pred),       2),
    }


# Helper: tune + train a model, then evaluate it
def train_with_tuning(model, param_grid, name,
                      X_tr, y_tr, X_v, y_v, X_te, y_te):
    print(f"Training: {name} ...", end=" ", flush=True)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    if param_grid:
        searcher = RandomizedSearchCV(
            estimator=model, param_distributions=param_grid,
            n_iter=10, cv=cv, scoring="roc_auc",
            n_jobs=1, random_state=42, verbose=0
        )
        searcher.fit(X_tr, y_tr)
        best = searcher.best_estimator_
    else:
        best = model
        best.fit(X_tr, y_tr)

    metrics = evaluate_model(best, name, X_v, y_v, X_te, y_te)
    print(f"Test Acc={metrics['Test Accuracy (%)']:.2f}%  AUC={metrics['Test ROC-AUC']:.3f}")
    return best, metrics

print("Helper functions ready.")

In [ ]:
# Define all 7 models and the hyperparameter values to search over
models_to_train = {
    "Random Forest": (
        RandomForestClassifier(random_state=42, n_jobs=1),
        {"n_estimators": [100, 200, 300],
         "max_depth": [None, 10, 20],
         "min_samples_split": [2, 5, 10]}
    ),
    "XGBoost": (
        xgb.XGBClassifier(eval_metric="logloss", random_state=42,
                          use_label_encoder=False, n_jobs=1),
        {"n_estimators": [100, 200, 300],
         "max_depth": [3, 5, 7],
         "learning_rate": [0.01, 0.1, 0.2],
         "subsample": [0.8, 1.0]}
    ),
    "LightGBM": (
        lgb.LGBMClassifier(random_state=42, verbose=-1, n_jobs=1),
        {"n_estimators": [100, 200, 300],
         "max_depth": [-1, 5, 10],
         "learning_rate": [0.01, 0.1, 0.2],
         "num_leaves": [31, 63, 127]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"max_depth": [3, 5, 10, None],
         "min_samples_split": [2, 5, 10],
         "criterion": ["gini", "entropy"]}
    ),
    "Linear SVM": (
        LinearSVC(random_state=42),
        {"C": [0.1, 1, 10]}
    ),
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        {"C": [0.01, 0.1, 1, 10]}
    ),
    "Naive Bayes": (
        GaussianNB(),
        None   # no hyperparameters to tune
    ),
}

trained_models = {}
results_list   = []

for model_name, (model_obj, param_grid) in models_to_train.items():
    best_model, metrics = train_with_tuning(
        model_obj, param_grid, model_name,
        X_train_bal, y_train_bal,
        X_val, y_val,
        X_test, y_test,
    )
    trained_models[model_name] = best_model
    results_list.append(metrics)

## Step 7 — Ensemble Model (RF + XGBoost + LightGBM)

**Soft voting** = average the predicted probabilities of all 3 models.  
This is smarter than hard voting (majority wins) because it considers *how confident* each model is.

In [ ]:
ensemble = VotingClassifier(
    estimators=[
        ("rf",  trained_models["Random Forest"]),
        ("xgb", trained_models["XGBoost"]),
        ("lgb", trained_models["LightGBM"]),
    ],
    voting="soft",   # use probability averages
    n_jobs=1,
)
ensemble.fit(X_train_bal, y_train_bal)

ens_metrics = evaluate_model(ensemble, "Ensemble Model",
                              X_val, y_val, X_test, y_test)
results_list.append(ens_metrics)
trained_models["Ensemble Model"] = ensemble

print(f"Ensemble → Test Acc={ens_metrics['Test Accuracy (%)']:.2f}%  "
      f"AUC={ens_metrics['Test ROC-AUC']:.3f}")

## Step 8 — Model Comparison (Table II from paper)

In [ ]:
results_df = pd.DataFrame(results_list).set_index("Model")
results_df

In [ ]:
# Save to CSV
results_df.to_csv("model_comparison.csv")

# Bar chart — Test Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_df["Test Accuracy (%)"].plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="black"
)
axes[0].set_title("Test Accuracy (%)")
axes[0].set_ylabel("%")
axes[0].set_xticklabels(results_df.index, rotation=30, ha="right")
axes[0].set_ylim(65, 100)

results_df["Test ROC-AUC"].plot(
    kind="bar", ax=axes[1], color="tomato", edgecolor="black"
)
axes[1].set_title("Test ROC-AUC")
axes[1].set_ylabel("AUC")
axes[1].set_xticklabels(results_df.index, rotation=30, ha="right")
axes[1].set_ylim(0.7, 1.0)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()

---
# Explainability (XAI)

The next sections explain *why* the models make their predictions.

We use **XGBoost** as the base model for SHAP because `TreeExplainer` works fastest with tree-based models.

In [ ]:
# Use a small sample of the test set to keep SHAP fast
SAMPLE_SIZE    = 500
xgb_model      = trained_models["XGBoost"]
X_test_sample  = X_test.iloc[:SAMPLE_SIZE].reset_index(drop=True)
y_test_sample  = y_test.iloc[:SAMPLE_SIZE].reset_index(drop=True)

## Step 9 — SHAP (SHapley Additive exPlanations)

SHAP answers: **which features push the prediction towards diabetic (+) or away (-)?**  
It uses game theory to fairly share credit between all features.

In [ ]:
# Create SHAP explainer and compute SHAP values for all test samples
explainer_shap = shap.TreeExplainer(xgb_model)
shap_values    = explainer_shap.shap_values(X_test_sample)
shap_explanation = explainer_shap(X_test_sample)

print("SHAP values computed. Shape:", shap_values.shape)

In [ ]:
# --- SHAP Summary Plot (Fig. 2 from paper) ---
# Each dot = one patient.
# Red dot = patient had a HIGH value for that feature.
# Blue dot = patient had a LOW value.
# Dot on the RIGHT = pushed the model towards predicting diabetic.
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample,
                  feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot — Global Feature Importance (Fig. 2)")
plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=150)
plt.show()

In [ ]:
# --- SHAP Waterfall Plot (Fig. 7) — one patient explained ---
# Starts from the average prediction (base value).
# Red bars push prediction UP (towards diabetic).
# Blue bars push prediction DOWN (towards healthy).
shap.plots.waterfall(shap_explanation[0], show=False)
plt.title("SHAP Waterfall — Patient 0 (Fig. 7)")
plt.tight_layout()
plt.savefig("shap_waterfall.png", dpi=150)
plt.show()

In [ ]:
# --- SHAP Force Plot (Fig. 6) ---
# Same idea as waterfall but shown as a horizontal push/pull bar.
shap.plots.force(shap_explanation[0], matplotlib=True, show=False)
plt.title("SHAP Force Plot — Patient 0 (Fig. 6)")
plt.tight_layout()
plt.savefig("shap_force_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- SHAP Decision Plot (Fig. 8) — 20 patients at once ---
# Each line = one patient's path from base value to final prediction.
plt.figure(figsize=(10, 8))
shap.decision_plot(
    explainer_shap.expected_value,
    shap_values[:20],
    X_test_sample.iloc[:20],
    feature_names=feature_names,
    show=False,
)
plt.title("SHAP Decision Plot — 20 Patients (Fig. 8)")
plt.tight_layout()
plt.savefig("shap_decision_plot.png", dpi=150)
plt.show()

In [ ]:
# Top-5 features by average SHAP importance
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = (
    pd.DataFrame({"Feature": feature_names, "Mean |SHAP|": mean_abs_shap})
    .sort_values("Mean |SHAP|", ascending=False)
)
print("Top 5 features by SHAP:")
shap_importance.head(5)

## Step 10 — Permutation Importance (Fig. 4 from paper)

Shuffle one feature at a time, measure how much accuracy drops.  
**Big drop = that feature was very important.**

In [ ]:
perm = permutation_importance(
    ensemble, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=1
)
perm_df = (
    pd.DataFrame({"Feature": feature_names,
                  "Mean Decrease in Accuracy": perm.importances_mean})
    .sort_values("Mean Decrease in Accuracy", ascending=True)
)

plt.figure(figsize=(10, 8))
plt.barh(perm_df["Feature"], perm_df["Mean Decrease in Accuracy"],
         color="steelblue")
plt.xlabel("Mean Decrease in Accuracy")
plt.title("Permutation Importance — Ensemble Model (Fig. 4)")
plt.tight_layout()
plt.savefig("permutation_importance.png", dpi=150)
plt.show()

## Step 11 — EBM (Explainable Boosting Machine) — Fig. 3 from paper

EBM is an inherently interpretable model — it is accurate like a boosted model  
but you can read exactly what each feature contributes (no black box needed).

In [ ]:
# Train EBM
ebm = ExplainableBoostingClassifier(random_state=42)
ebm.fit(X_train_bal, y_train_bal)

ebm_pred = np.asarray(ebm.predict(X_test)).astype(int)
y_test_arr = np.asarray(y_test).astype(int)

print("y_test unique:", np.unique(y_test_arr))
print("ebm_pred unique:", np.unique(ebm_pred))
print("y_test shape:", y_test_arr.shape)
print("ebm_pred shape:", ebm_pred.shape)

ebm_acc = accuracy_score(y_test_arr, ebm_pred) * 100
ebm_auc = roc_auc_score(y_test_arr, ebm.predict_proba(X_test)[:, 1])

print(f"EBM → Test Accuracy: {ebm_acc:.2f}%   ROC-AUC: {ebm_auc:.3f}")

In [ ]:
# EBM feature importance bar chart (Fig. 3)

ebm_importances = (
    pd.DataFrame({
        "Feature": ebm.term_names_,
        "Mean Abs Score": ebm.term_importances()
    })
    .sort_values("Mean Abs Score", ascending=True)
)

# 🔥 take top 10
top_features = ebm_importances.tail(10)

plt.figure(figsize=(10, 8))
plt.barh(top_features["Feature"], top_features["Mean Abs Score"],
         color="orange")

plt.xlabel("Mean Absolute Score (Weighted)")
plt.title("EBM Feature Importance — Top Global Predictors (Fig. 3)")
plt.tight_layout()
plt.savefig("ebm_feature_importance.png", dpi=150)
plt.show()

In [ ]:
# Interactive EBM explanation (opens in Jupyter output)
ebm_global = ebm.explain_global(name="EBM Global Importance")
ebm_show(ebm_global)

In [ ]:
# EBM local explanation for 5 individual patients
ebm_local = ebm.explain_local(
    X_test.iloc[:5], y_test.iloc[:5], name="EBM Local — 5 patients"
)
ebm_show(ebm_local)

## Step 12 — LIME (Local Interpretable Model-Agnostic Explanations) — Fig. 5 from paper

LIME explains **one prediction at a time** by building a simple, readable model  
in the neighbourhood of that specific data point.

In [ ]:
# Create LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data = X_train_bal.values,
    feature_names = feature_names,
    class_names   = ["Non-Diabetic", "Diabetic"],
    mode          = "classification",
    random_state  = 42,
)
print("LIME explainer ready.")

In [ ]:
# Explain Patient 0 (Fig. 5 from paper)
instance_0 = X_test.iloc[0].values
lime_exp   = lime_explainer.explain_instance(
    data_row   = instance_0,
    predict_fn = ensemble.predict_proba,
    num_features = 10,
)

fig = lime_exp.as_pyplot_figure()
plt.title("LIME Explanation — Patient 0 (Fig. 5)")
plt.tight_layout()
plt.savefig("lime_explanation_instance0.png", dpi=150)
plt.show()

print("\nTop 10 features for Patient 0:")
print(f"  Ensemble predicted class: {ensemble.predict([instance_0])[0]} "
      f"({'Diabetic' if ensemble.predict([instance_0])[0] == 1 else 'Non-Diabetic'})")
print()
for rule, weight in lime_exp.as_list():
    direction = "↑ towards DIABETIC" if weight > 0 else "↓ towards HEALTHY"
    print(f"  {rule:45s}  {weight:+.5f}  {direction}")

## Step 13 — Partial Dependence Plots (PDPs) — Fig. 9 from paper

PDPs show the **average effect** of one feature on the prediction while holding all other features at their average.  
We look at `HighBP` and `HighChol` — the paper shows both strongly increase diabetes risk.

In [ ]:
highbp_idx   = feature_names.index("HighBP")
highchol_idx = feature_names.index("HighChol")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

PartialDependenceDisplay.from_estimator(
    ensemble,
    X_test,
    features=[highbp_idx],
    feature_names=feature_names,
    ax=axes[0]
)

PartialDependenceDisplay.from_estimator(
    ensemble,
    X_test,
    features=[highchol_idx],
    feature_names=feature_names,
    ax=axes[1]
)

axes[0].set_title("HighBP")
axes[1].set_title("HighChol")

plt.suptitle("Partial Dependence — HighBP & HighChol vs. Diabetes Risk (Fig. 9)",
             fontsize=12)
plt.tight_layout()
plt.savefig("pdp_highbp_highchol.png", dpi=150)
plt.show()

## Step 14 — Anchor Explanations — Fig. 10 from paper

Anchors produce **IF-THEN rules**:  
> *IF BMI > 30 AND PhysActivity = 0  THEN  Diabetic (90% confidence)*

These are easy for doctors to read and act on.

In [ ]:
try:
    from alibi.explainers import AnchorTabular

    anchor_exp = AnchorTabular(
        predictor     = ensemble.predict,
        feature_names = feature_names,
    )
    anchor_exp.fit(X_train_bal.values, disc_perc=(25, 50, 75))

    anchor_result = anchor_exp.explain(X_test.iloc[0].values, threshold=0.90)
    print("Anchor Rule:")
    print(f"  IF  {' AND '.join(anchor_result.anchor)}")
    print(f"  THEN  Diabetic")
    print(f"  Precision : {anchor_result.precision:.2f}")
    print(f"  Coverage  : {anchor_result.coverage:.2f}")

except ImportError:
    print("alibi not installed — skipping anchor model.")
    print("Install with: pip install alibi[tensorflow]")

In [ ]:
# Plot PhysActivity distribution with threshold line (Fig. 10)
phys_vals = X_test["PhysActivity"]
threshold = float(phys_vals.median())

plt.figure(figsize=(8, 5))
plt.hist(phys_vals, bins=30, color="steelblue", alpha=0.7)
plt.axvline(threshold, color="red", linestyle="--",
            label=f"Anchor Threshold = {threshold:.2f}")
plt.xlabel("PhysActivity (scaled)")
plt.ylabel("Count")
plt.title("Distribution of PhysActivity with Anchor Threshold (Fig. 10)")
plt.legend()
plt.tight_layout()
plt.savefig("anchor_threshold_physactivity.png", dpi=150)
plt.show()

## Step 15 — Counterfactual Explanations — Table III from paper

Counterfactuals answer: **"What would you need to change to NOT be diabetic?"**  
e.g. *Lower your BMI by 0.24, increase physical activity by 0.57*

In [ ]:
try:
    import dice_ml
    from dice_ml import Dice

    # DiCE needs a single dataframe with features + label
    train_df_dice = X_train_bal.copy()
    train_df_dice[TARGET] = y_train_bal.values

    dice_data  = dice_ml.Data(
        dataframe           = train_df_dice,
        continuous_features = feature_names,
        outcome_name        = TARGET,
    )
    dice_model = dice_ml.Model(model=ensemble, backend="sklearn")
    dice_exp   = Dice(dice_data, dice_model, method="random")

    # Find the first diabetic patient in the test set
    diabetic_instance = X_test[y_test == 1].iloc[:1]

    cf_result = dice_exp.generate_counterfactuals(
        diabetic_instance,
        total_CFs     = 3,
        desired_class = "opposite",   # flip Diabetic → Non-Diabetic
    )
    print("Counterfactual explanations (only the changed features are shown):")
    cf_result.visualize_as_dataframe(show_only_changes=True)

except ImportError:
    print("dice-ml not installed — showing paper values instead.")
    print("Install with: pip install dice-ml")

In [ ]:
# Reference values from paper — Table III
cf_paper = pd.DataFrame([
    {"HighBP":1.15,"HighChol":1.17,"CholCheck":-2.89,"BMI":0.24,
     "Smoker":-0.89,"Stroke":-0.21,"HeartDiseaseorAttack":-0.32,
     "PhysActivity":0.57,"Fruits":-1.32,"Veggies":-0.32,
     "HvyAlcoholConsump":-0.24,"AnyHealthcare":0.23,
     "NoDocbcCost":-0.30,"GenHlth":1.39,"MentHlth":-0.16},
    {"HighBP":1.15,"HighChol":1.17,"CholCheck":0.20,"BMI":0.24,
     "Smoker":-0.89,"Stroke":-0.21,"HeartDiseaseorAttack":-0.32,
     "PhysActivity":0.57,"Fruits":-1.32,"Veggies":-2.07,
     "HvyAlcoholConsump":-0.24,"AnyHealthcare":0.23,
     "NoDocbcCost":-0.30,"GenHlth":1.39,"MentHlth":-0.16},
    {"HighBP":1.15,"HighChol":-0.21,"CholCheck":0.20,"BMI":0.24,
     "Smoker":-0.89,"Stroke":1.26,"HeartDiseaseorAttack":-0.32,
     "PhysActivity":0.57,"Fruits":-1.32,"Veggies":-2.07,
     "HvyAlcoholConsump":-0.24,"AnyHealthcare":0.23,
     "NoDocbcCost":-0.30,"GenHlth":1.39,"MentHlth":-0.16},
])
print("Counterfactual coefficients from paper (Table III):")
cf_paper

## Step 16 — Explainability Metrics (Table IV from paper)

These metrics measure **how good the explanations themselves are**, not the model accuracy.

| Metric | What it means | Good value |
|--------|--------------|------------|
| **Fidelity** | LIME's prediction matches ensemble's prediction | Close to 1 |
| **Faithfulness** | Removing top features actually changes the prediction | Higher is better |
| **Sparsity** | Few features used per explanation (simpler = better) | Lower number |
| **Stability** | Same patient gets same explanation every time | Close to 0 |
| **Consistency** | SHAP and LIME agree on the most important features | Close to 1 |

In [ ]:
METRIC_SAMPLE = 100   # number of patients to use for metric calculation

# --- FIDELITY ---
# How closely does LIME match the ensemble's probability output?
fidelity_scores = []
for i in range(METRIC_SAMPLE):
    instance = X_test.iloc[i].values
    exp      = lime_explainer.explain_instance(
        instance, ensemble.predict_proba, num_features=len(feature_names)
    )
    lime_prob     = exp.local_pred[0]
    ensemble_prob = ensemble.predict_proba([instance])[0][1]
    fidelity_scores.append(1.0 - abs(lime_prob - ensemble_prob))
fidelity = float(np.mean(fidelity_scores))
print(f"Fidelity computed: {fidelity:.3f}")

In [ ]:
# --- FAITHFULNESS ---
# Mask the top-5 SHAP features (set to 0) — if prediction changes a lot,
# the explanation was faithfully pointing at truly important features.
def compute_faithfulness(model, X_sample, shap_vals, top_k=5):
    scores = []
    for i in range(len(X_sample)):
        orig_prob = model.predict_proba(X_sample.iloc[[i]])[0][1]
        top_idx   = np.argsort(np.abs(shap_vals[i]))[-top_k:]
        X_masked  = X_sample.iloc[[i]].copy()
        X_masked.iloc[0, top_idx] = 0.0
        new_prob  = model.predict_proba(X_masked)[0][1]
        scores.append(abs(orig_prob - new_prob))
    return float(np.mean(scores))

faithfulness = compute_faithfulness(xgb_model, X_test_sample, shap_values)
print(f"Faithfulness computed: {faithfulness:.3f}")

In [ ]:
# --- SPARSITY ---
# How many features does LIME actually use per explanation?
sparsity_counts = []
for i in range(METRIC_SAMPLE):
    instance = X_test.iloc[i].values
    exp      = lime_explainer.explain_instance(
        instance, ensemble.predict_proba, num_features=len(feature_names)
    )
    used = sum(1 for _, w in exp.as_list() if abs(w) > 1e-4)
    sparsity_counts.append(used)
sparsity = float(np.mean(sparsity_counts))
print(f"Sparsity computed: {sparsity:.1f} features on average")

In [ ]:
# --- STABILITY ---
# Run LIME twice on the same patient and compare the explanations.
# Close to 0 = very stable (same answer every time).
stability_diffs = []
for i in range(20):
    instance = X_test.iloc[i].values
    exp1 = lime_explainer.explain_instance(instance, ensemble.predict_proba, num_features=10)
    exp2 = lime_explainer.explain_instance(instance, ensemble.predict_proba, num_features=10)
    w1   = dict(exp1.as_list())
    w2   = dict(exp2.as_list())
    common = set(w1) & set(w2)
    if common:
        stability_diffs.append(np.mean([abs(w1[k] - w2[k]) for k in common]))
stability = float(np.mean(stability_diffs)) if stability_diffs else 0.0
print(f"Stability computed: {stability:.2e}")

In [ ]:
# --- CONSISTENCY ---
# Do SHAP and LIME agree on the top-5 most important features?
shap_top5 = set(
    pd.DataFrame({"Feature": feature_names, "Score": mean_abs_shap})
    .nlargest(5, "Score")["Feature"]
)

lime_weights_dict = dict(lime_exp.as_list())
lime_top5_raw     = sorted(lime_weights_dict,
                           key=lambda k: abs(lime_weights_dict[k]),
                           reverse=True)[:5]

def extract_feature_name(rule_string):
    """LIME rules look like '0.08 < BMI <= 0.70' — extract 'BMI'."""
    for feat in feature_names:
        if feat in rule_string:
            return feat
    return rule_string

lime_top5 = {extract_feature_name(k) for k in lime_top5_raw}
consistency = len(shap_top5 & lime_top5) / 5.0

print(f"SHAP top-5:  {shap_top5}")
print(f"LIME top-5:  {lime_top5}")
print(f"Consistency: {consistency:.2f}")

In [ ]:
# --- Summary Table (Table IV from paper) ---
metrics_table = pd.DataFrame([
    {"Metric": "Fidelity",
     "Value": round(fidelity, 3),
     "Meaning": "LIME matches ensemble predictions (closer to 1.0 = better)"},
    {"Metric": "Faithfulness",
     "Value": round(faithfulness, 3),
     "Meaning": "Removing top features changes prediction (higher = better)"},
    {"Metric": "Sparsity",
     "Value": round(sparsity, 1),
     "Meaning": "Avg features per explanation (lower = simpler)"},
    {"Metric": "Stability",
     "Value": f"{stability:.2e}",
     "Meaning": "Explanation difference on same input (closer to 0 = more stable)"},
    {"Metric": "Consistency",
     "Value": round(consistency, 2),
     "Meaning": "SHAP and LIME agree on top features (closer to 1 = better)"},
]).set_index("Metric")

print("Explainability Metrics (Table IV from paper):")
metrics_table.to_csv("explainability_metrics.csv")
metrics_table

---
## Final Summary

In [ ]:
print("=" * 60)
print("MODEL PERFORMANCE (Table II)")
print("=" * 60)
print(results_df.to_string())

print("\n" + "=" * 60)
print("TOP 5 PREDICTORS (SHAP global importance)")
print("=" * 60)
print(shap_importance.head(5).to_string(index=False))

print("\n" + "=" * 60)
print("EXPLAINABILITY METRICS (Table IV)")
print("=" * 60)
print(metrics_table[["Value", "Meaning"]].to_string())

print("\nOutput files saved:")
for f in ["model_comparison.csv", "model_comparison.png",
          "shap_summary_plot.png", "shap_waterfall.png",
          "shap_force_plot.png", "shap_decision_plot.png",
          "permutation_importance.png", "ebm_feature_importance.png",
          "lime_explanation_instance0.png", "pdp_highbp_highchol.png",
          "anchor_threshold_physactivity.png", "explainability_metrics.csv"]:
    print(f"  {f}")